# 🎯 Stack 2.9 — 128K Context Fine-tuning
Fine-tune Qwen2.5-Coder-1.5B from 32K → 128K context

**Runtime:** GPU (T4 16GB recommended) | **Time:** ~2-3 hours

## Step 1: Clone Stack 2.9 & Install Dependencies

In [ ]:
import os
os.chdir("/content")

# Clone repo
!git clone https://github.com/my-ai-stack/stack-2.9.git

# Install dependencies
!pip install -q transformers>=4.40.0 peft datasets bitsandbytes accelerate huggingface_hub
!pip install -q scipy torch --upgrade
!pip install -q flash-attn --no-build-isolation  # Optional: for faster attention

## Step 2: Login to HuggingFace (push weights later)

In [ ]:
from huggingface_hub import login

# Get your token at: https://huggingface.co/settings/tokens
# Replace with your actual token before running
HF_TOKEN = "YOUR_HF_TOKEN"  # ← Replace with your HF token
login(token=HF_TOKEN)
print("✓ Logged in to HuggingFace")

## Step 3: Mount Google Drive (optional — for saving checkpoints)

In [ ]:
# Check GPU and environment
import torch
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Mount Google Drive (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# OUTPUT_DIR = "/content/drive/MyDrive/stack-2.9-128k-output"

OUTPUT_DIR = "/content/stack-2.9/output/stack-2.9-128k"

## Step 3.5: Verify GPU Setup

import os
os.chdir("/content/stack-2.9")

# Set environment for better GPU usage
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"

# === UPDATE THESE VALUES BEFORE RUNNING ===
YOUR_HF_USERNAME = "your-username"  # Replace with your HF username

# Run training
!python training/train_extended_context.py \
    --model-path Qwen/Qwen2.5-Coder-1.5B \
    --data-path training/training-data/tool_examples_combined.jsonl \
    --output-dir /content/stack-2.9/output/stack-2.9-128k \
    --context-length 131072 \
    --lora-rank 64 \
    --epochs 3 \
    --batch-size 1 \
    --grad-accum 16 \
    --push-to-hub \
    --hub-model-id YOUR_HF_USERNAME/stack-2.9-128k

In [ ]:
import os
import sys
os.chdir("/content/stack-2.9")

# Set environment for better GPU usage
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "true"

# Training parameters - UPDATE THESE
YOUR_HF_TOKEN = "YOUR_HF_TOKEN"  # Replace with your token
YOUR_USERNAME = "your-username"   # Replace with your HF username

# Run training directly for better output visibility
!python training/train_extended_context.py \
    --model-path Qwen/Qwen2.5-Coder-1.5B \
    --data-path training/training-data/tool_examples_combined.jsonl \
    --output-dir /content/stack-2.9/output/stack-2.9-128k \
    --context-length 131072 \
    --lora-rank 64 \
    --epochs 3 \
    --batch-size 1 \
    --grad-accum 16 \
    --push-to-hub \
    --hub-model-id {YOUR_USERNAME}/stack-2.9-128k

## Step 5: Download Model from Colab (if mounted Drive)

In [ ]:
# Check output
!ls -la /content/stack-2.9/output/stack-2.9-128k/merged/